In [ ]:
import os
import subprocess
from huggingface_hub import snapshot_download

MODEL_NAME = "julianpins/tinyllama-1.1b-chat-numpy-refactor-v1"

def setup_llama_cpp():
    print("Setting up llama.cpp")

    if not os.path.exists("llama.cpp"):
        print("Cloning llama.cpp")
        subprocess.run(["git", "clone", "https://github.com/ggml-org/llama.cpp"], check=True)

    os.chdir("llama.cpp")

    subprocess.run(["cmake", "-B", "build"], check=True)
    subprocess.run(["cmake", "--build", "build", "--config", "Release", "-j", "8"], check=True)

    subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)

    # Copy needed binaries
    os.chdir("build/bin/")
    subprocess.run(["cp", "llama-quantize", "../../"], check=True)
    os.chdir("../..")

    os.chdir("..")
    print("DONE: llama.cpp setup complete!")


def convert_tinyllama():
    print(f"Converting: {MODEL_NAME}")

    folder_name = MODEL_NAME.replace("/", "_")
    model_dir = f"models/{folder_name}"

    # Step 1: Download model
    print(f"Downloading {MODEL_NAME}...")
    if not os.path.exists(model_dir):
        os.makedirs("models", exist_ok=True)
        try:
            snapshot_download(
                repo_id=MODEL_NAME,
                local_dir=model_dir
            )
            # Clean up problematic directories
            huggingface_dir = os.path.join(model_dir, ".huggingface")
            if os.path.exists(huggingface_dir):
                import shutil
                shutil.rmtree(huggingface_dir, ignore_errors=True)
        except Exception as e:
            print(f"Download error: {e}")
            return

    # Step 2: Convert to f16 GGUF
    print(f"\nConverting to f16 GGUF:")
    f16_file = f"{folder_name}_f16.gguf"
    if not os.path.exists(f16_file):
        cmd = [
            "python", "llama.cpp/convert_hf_to_gguf.py",
            model_dir,
            "--outtype", "f16",
            "--outfile", f16_file
        ]
        print(f"Running: {' '.join(cmd)}")
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            print("DONE: Conversion to f16 successful")
            if result.stdout:
                print(f"STDOUT: {result.stdout}")
        except subprocess.CalledProcessError as e:
            print(f"Conversion failed with exit code {e.returncode}")
            print(f"STDOUT: {e.stdout}")
            print(f"STDERR: {e.stderr}")
            return

    # Check f16 file size
    if os.path.exists(f16_file):
        f16_size = os.path.getsize(f16_file) / (1024 * 1024)
        print(f"f16 GGUF size: {f16_size:.1f} MB")
        if f16_size < 2000:  # Should be ~2.2GB for TinyLlama
            print("WARNING: f16 file seems too small!")

    # Step 3: Quantize to q4_k_m
    print(f"\nQuantizing to q4_k_m...")
    q4_file = f"{folder_name}_q4_k_m.gguf"
    if not os.path.exists(q4_file):
        cmd = [
            "./llama.cpp/llama-quantize",
            f16_file,
            q4_file,
            "q4_k_m"
        ]
        print(f"Running: {' '.join(cmd)}")
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=True)
            print("DONE: Quantization successful")
            if result.stdout:
                print(f"Quantization output: {result.stdout}")
        except subprocess.CalledProcessError as e:
            print(f"Quantization failed with exit code {e.returncode}")
            print(f"STDERR: {e.stderr}")
            return

    # Final file sizes
    if os.path.exists(f16_file):
        f16_size = os.path.getsize(f16_file) / (1024 * 1024)
        print(f"f16 size: {f16_size:.1f} MB")

    if os.path.exists(q4_file):
        q4_size = os.path.getsize(q4_file) / (1024 * 1024)
        print(f"q4_k_m size: {q4_size:.1f} MB")
        print(f"Final file: {q4_file}")

        # Sanity check
        if q4_size < 500:
            print("WARNING: Final quantized file is suspiciously small!")
            print("Expected size for TinyLlama q4_k_m: ~600-700 MB")
        else:
            print("File size looks reasonable for TinyLlama!")

def main():
    # Setup
    setup_llama_cpp()

    # Convert with debugging
    convert_tinyllama()

    print(f"\nDONE: Process complete!")

if __name__ == "__main__":
    main()